# Tensor Product Block Encoding

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from dataclasses import replace

import torch
from IPython.display import SVG, display
from sympy import symbols

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.resource_model import ComponentCost, GateCost
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.register_types import opaque_register_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.simulation.unitary import get_unitary

## Tensor products of block encodings

The encoding of $U_1 \otimes \cdots \otimes U_n$ is the tensor product
of its child encodings. The scale is $\prod_i \alpha_i$, the ancilla width
is $\sum_i a_i$, and the error bound is $\sum_i \alpha_i \epsilon_i$.

The system, ancilla and resource ports each concatenate child words in
tuple order. Omit zero-width ports. The caller supplies per-child work
and measurement slots, and a child can reuse clean work released by an
earlier child. Symbolic auxiliary widths below describe the signature.
Executable binding requires concrete port widths.

In [ ]:
from flagquantum.algorithms.primitives.tensor_product_block_encoding import (
    TensorProductBlockEncoding,
)
from flagquantum.algorithms.primitives.unitary_block_encoding import UnitaryBlockEncoding


def signature(system, ancilla=0, resource=0):
    return Signature(tuple(
        Register(name, opaque_register_type(width))
        for name, width in (("system", system), ("ancilla", ancilla), ("resource", resource))
        if width != 0
    ))


def block_signature(encoding):
    return signature(encoding.system_bitsize, encoding.ancilla_bitsize, encoding.resource_bitsize)

### Example Instances

In [ ]:
def phase(ports, work, slots, adjoint):
    return (Instruction("tdg" if adjoint else "t", ports["q"]),)


def hadamard(ports, work, slots, adjoint):
    return (Instruction("h", ports["q"]),)


t = UnitaryBlockEncoding((("q", 1),), phase)
h = UnitaryBlockEncoding((("q", 1),), hadamard)
plain = TensorProductBlockEncoding((t, h))
assert (plain.alpha, plain.epsilon, plain.system_bitsize) == (1, 0, 2)

In [ ]:
def controlled_not(ports, work, slots, adjoint):
    return (Instruction("cx", (*ports["control"], *ports["target"])),)


cx = UnitaryBlockEncoding((("control", 1), ("target", 1)), controlled_not)
augmented_t = replace(t, alpha=0.5, ancilla_bitsize=2, resource_bitsize=1, epsilon=0.01)
augmented_cx = replace(cx, alpha=0.5, ancilla_bitsize=1, resource_bitsize=1, epsilon=0.1)
overrides = TensorProductBlockEncoding((augmented_t, augmented_cx))
assert (overrides.system_bitsize, overrides.ancilla_bitsize, overrides.resource_bitsize) == (3, 3, 2)
assert (overrides.alpha, overrides.epsilon) == (0.25, 0.055)

In [ ]:
alpha1, alpha2, a1, a2, eps1, eps2 = symbols("alpha1 alpha2 a1 a2 eps1 eps2")
symbolic = TensorProductBlockEncoding((
    replace(t, alpha=alpha1, epsilon=eps1),
    replace(h, alpha=alpha2, epsilon=eps2),
))
assert symbolic.alpha == alpha1 * alpha2
assert symbolic.epsilon == alpha1 * eps1 + alpha2 * eps2
symbolic_signature = signature(symbolic.system_bitsize, a1 + a2)
print("Symbolic alpha and epsilon", symbolic.alpha, symbolic.epsilon)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("T and Hadamard", block_signature(plain), {}),
    ("T and CNOT with overrides", block_signature(overrides), {}),
    ("Symbolic T and Hadamard", symbolic_signature, {}),
])))

### Call Graph

In [ ]:
t_cost = t.resource_model(ComponentCost("T", 1, GateCost(t=1)))
h_cost = h.resource_model(ComponentCost("Hadamard", 1, GateCost(clifford=1)))
model = plain.resource_model((replace(t_cost, name="Unitary(T)"),
                              replace(h_cost, name="Unitary(Hadamard)")))
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
print([(call.component.name, call.repetitions) for call in model.calls])

### Append the tensor product and its adjoint

Bind both children to their assigned system words. The complete matrix
checks the relative phase on the first qubit and the sign of the second
qubit's Hadamard amplitude.

In [ ]:
def bind(child):
    def apply(ports, work, slots, adjoint):
        return child.make_on(system=ports["system"], ancilla=ports["ancilla"],
                             resource=ports["resource"], workspace=work,
                             classical_bits=slots, adjoint=adjoint)
    return apply


binders = (bind(t), bind(h))
ports = dict(system=(0, 1), workspaces=((), ()), classical_bits=((), ()))
circuit = Circuit(2, dtype=torch.complex128).append(plain.make_on(binders, **ports))
t_matrix = torch.diag(torch.tensor([1, complex(2**-0.5, 2**-0.5)], dtype=torch.complex128))
h_matrix = torch.tensor([[1, 1], [1, -1]], dtype=torch.complex128) / 2**0.5
torch.testing.assert_close(get_unitary(circuit), torch.kron(t_matrix, h_matrix))
circuit.append(plain.make_on(binders, **ports, adjoint=True))
torch.testing.assert_close(get_unitary(circuit), torch.eye(4, dtype=torch.complex128))